# 11. 주파수 이상치 탐지 (f)

## 이상치 기준
- 물리적 기준: 유럽 전력망 주파수 규격 EN 50160 기준 50Hz ± 1% = 49.5~50.5Hz 범위 초과
- 통계적 기준: 계량기별 일별 평균값 기준 평균 ± 3σ 초과

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

# EN 50160: 50Hz ± 1%
F_LOWER = 49.5
F_UPPER = 50.5

ALL_METERS = [
    'H1.Z10', 'H1.Z11', 'H1.Z12', 'H1.Z13', 'H1.Z14', 'H1.Z15', 'H1.Z16',
    'H1.Z17', 'H1.Z18', 'H1.Z19', 'H1.Z20', 'H1.Z21', 'H1.Z22', 'H1.Z23',
    'H1.Z24', 'H1.Z25', 'H1.Z26', 'H1.Z27', 'H1.Z28', 'H1.Z29', 'H1.Z310',
    'H1.ZE20', 'H2.T.Z30', 'H2.T.Z31', 'H2.T.Z32', 'H2.T.Z33', 'H2.T.Z34',
    'H2.Z311', 'H2.Z35', 'H2.Z64', 'H2.Z65', 'H2.Z66', 'H2.Z67', 'H2.Z68',
    'H2.Z69', 'H2.Z70', 'H2.ZE64', 'H2.ZE65', 'H2.ZE66', 'H2.ZE67', 'H2.ZE74',
    'H3.Z312', 'H3.Z40', 'H3.Z41', 'H3.Z42', 'H4.Z50', 'H4.Z51', 'H4.ZE50',
    'H4.ZE51', 'V.Z81', 'V.Z82', 'V.Z84', 'V.ZE84'
]

save_dir = ROOT / 'outputs/tables/anomaly'
save_dir.mkdir(parents=True, exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    df['day'] = pd.to_datetime(df['day'])
    return df


def detect_anomaly(meter, measurement):
    df = fetch_daily(meter, measurement)
    if df.empty:
        return None

    results = []

    # 1. 물리적 기준: EN 50160 49.5~50.5Hz 범위 초과
    physical = df[(df['min_val'] < F_LOWER) | (df['max_val'] > F_UPPER)].copy()
    physical['anomaly_type'] = '물리적이상(주파수범위초과)'
    physical['criterion'] = f'min_val < {F_LOWER}Hz 또는 max_val > {F_UPPER}Hz (EN 50160 50Hz±1%)'
    if len(physical) > 0:
        results.append(physical)

    # 2. 통계적 기준: 평균 ± 3σ
    mean_val = df['avg_val'].mean()
    std_val  = df['avg_val'].std()
    upper = mean_val + 3 * std_val
    lower = mean_val - 3 * std_val
    stat = df[(df['avg_val'] > upper) | (df['avg_val'] < lower)].copy()
    stat['anomaly_type'] = '통계적이상(3sigma)'
    stat['criterion'] = f'mean={mean_val:.4f}, sigma={std_val:.4f}, lower={lower:.4f}, upper={upper:.4f}'
    if len(stat) > 0:
        results.append(stat)

    if not results:
        return None

    result = pd.concat(results).drop_duplicates('day').sort_values('day')
    result['meter'] = meter
    result['measurement'] = measurement
    return result[['meter', 'measurement', 'day', 'min_val', 'max_val', 'avg_val', 'anomaly_type', 'criterion']]

In [3]:
all_results = []

for meter in ALL_METERS:
    result = detect_anomaly(meter, 'f')
    if result is not None and len(result) > 0:
        print(f'{meter} f: {len(result)}건')
        all_results.append(result)

if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_freq.csv', index=False)
    print(f'\n총 {len(final)}건 저장 완료')
else:
    print('이상치 없음')

/tmp/ipykernel_110505/1122178182.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 f: 16건


/tmp/ipykernel_110505/1122178182.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z11 f: 7건


/tmp/ipykernel_110505/1122178182.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


...

V.Z81 f: 17건


/tmp/ipykernel_110505/1122178182.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z82 f: 17건


/tmp/ipykernel_110505/1122178182.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_110505/1122178182.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.ZE84 f: 1건

총 463건 저장 완료


In [4]:
if all_results:
    summary = final.groupby(['meter', 'measurement', 'anomaly_type']).agg(
        건수=('day', 'count'),
        시작일=('day', 'min'),
        종료일=('day', 'max'),
        min_val=('min_val', 'min'),
        max_val=('max_val', 'max'),
        criterion=('criterion', 'first')
    ).reset_index()
    summary.to_csv(save_dir / 'anomaly_freq_summary.csv', index=False)
    print(summary.to_string())

       meter measurement    anomaly_type  건수        시작일        종료일    min_val    max_val                                                 criterion
0     H1.Z10           f   통계적이상(3sigma)  16 2018-02-07 2018-03-02  49.964333  50.025000  mean=50.0915, sigma=0.0287, lower=50.0055, upper=50.1775
1     H1.Z11           f  물리적이상(주파수범위초과)   1 2020-09-17 2020-09-17  49.486091  50.095083   min_val < 49.5Hz 또는 max_val > 50.5Hz (EN 50160 50Hz±1%)
2     H1.Z11           f   통계적이상(3sigma)   6 2018-02-26 2019-01-10  49.959917  50.018917  mean=50.0465, sigma=0.0213, lower=49.9826, upper=50.1104
3     H1.Z12           f  물리적이상(주파수범위초과)   1 2020-09-17 2020-09-17  49.054126  50.114500   min_val < 49.5Hz 또는 max_val > 50.5Hz (EN 50160 50Hz±1%)
4     H1.Z12           f   통계적이상(3sigma)   5 2018-02-26 2018-03-02  49.928583  49.988083  mean=50.0441, sigma=0.0299, lower=49.9544, upper=50.1338
5     H1.Z13           f   통계적이상(3sigma)   3 2018-02-28 2018-03-02  49.935750  49.986583  mean=50.0591, sigma=0.0330, 